# Práctica Lynis – Isolation Forest
**Celda 1:** lee tu `lynis.log` desde GitHub y lo convierte a CSV (crea el `df`).
**Celda 2:** código del profe (Isolation Forest).

In [5]:
import re
import urllib.request
import numpy as np
import pandas as pd

USUARIO = "abrilalejandragomezlerma6-ai"
HOST = "abril"
URL = f"https://raw.githubusercontent.com/{USUARIO}/Linux3B/main/Lynis/logs/{HOST}/lynis.log"

texto = urllib.request.urlopen(URL).read().decode("utf-8", errors="replace")

# Cada línea del log empieza con fecha y hora: 2026-10-09 09:40:12 mensaje
patron = re.compile(r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) (.*)$")
filas = []
for linea in texto.splitlines():
    m = patron.match(linea)
    if m:
        filas.append({"timestamp": m.group(1), "mensaje": m.group(2)})

df = pd.DataFrame(filas)
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Columnas numéricas que usará Isolation Forest
df["segundos_desde_inicio"] = (df["timestamp"] - df["timestamp"].min()).dt.total_seconds()
df["duracion_s"] = (df["timestamp"].shift(-1) - df["timestamp"]).dt.total_seconds().fillna(0)
df["longitud_mensaje"] = df["mensaje"].str.len()
df["es_warning"] = df["mensaje"].str.contains("warning", case=False).astype(int)
df["es_suggestion"] = df["mensaje"].str.contains("suggestion", case=False).astype(int)
df["es_test"] = df["mensaje"].str.contains("Performing test ID", case=False).astype(int)

df.to_csv("lynis_log.csv", index=False)
print("CSV generado: lynis_log.csv", df.shape)
df.head()

CSV generado: lynis_log.csv (10414, 8)


,timestamp,mensaje,segundos_desde_inicio,duracion_s,longitud_mensaje,es_warning,es_suggestion,es_test
0,2026-10-09 19:41:03,"Starting Lynis 3.0.9 with PID 57294, build dat...",0.0,0.0,58,0,0,0
1,2026-10-09 19:41:03,====,0.0,0.0,4,0,0,0
2,2026-10-09 19:41:03,"### 2007-2021, CISOfy - https://cisofy.com/lyn...",0.0,0.0,53,0,0,0
3,2026-10-09 19:41:03,Checking permissions of /usr/share/lynis/inclu...,0.0,0.0,57,0,0,0
4,2026-10-09 19:41:03,File permissions are OK,0.0,0.0,23,0,0,0


In [6]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.exceptions import NotFittedError

print("Dimensión original del dataset:", df.shape)

result_cols = ["anomaly_score", "is_anomaly"]
features = df.drop(columns=result_cols, errors="ignore")

X = features.select_dtypes(include=[np.number]).copy()

X = X.replace([np.inf, -np.inf], np.nan)

if X.shape[1] == 0:
    raise ValueError(
        "No existen columnas numéricas para entrenar IsolationForest."
    )

X = X.dropna(axis=1, how="all")

if X.shape[1] == 0:
    raise ValueError("Todas las columnas numéricas están vacías.")

print("Características utilizadas:", X.columns.tolist())
print("Dimensión de X:", X.shape)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("isolation_forest", IsolationForest(
        n_estimators=200,
        contamination=0.05,
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(X)

df["anomaly_score"] = model.decision_function(X)
df["is_anomaly"] = model.predict(X)

df["anomaly_label"] = df["is_anomaly"].map({
    1: "Normal",
    -1: "Anomalía"
})

print("\nDistribución de resultados:")
print(df["anomaly_label"].value_counts())

print("\nPorcentaje de anomalías:")
print(
    df["anomaly_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nPrimeras observaciones:")
print(df.head(10).to_string(index=False))

df.to_csv("isolation_forest_resultados.csv", index=False)

print("\nArchivo generado: isolation_forest_resultados.csv")

Dimensión original del dataset: (10414, 8)
Características utilizadas: ['segundos_desde_inicio', 'duracion_s', 'longitud_mensaje', 'es_warning', 'es_suggestion', 'es_test']
Dimensión de X: (10414, 6)

Distribución de resultados:
anomaly_label
Normal      9893
Anomalía     521
Name: count, dtype: int64

Porcentaje de anomalías:
anomaly_label
Normal      95.0
Anomalía     5.0
Name: proportion, dtype: float64

Primeras observaciones:
          timestamp                                                                    mensaje  segundos_desde_inicio  duracion_s  longitud_mensaje  es_warning  es_suggestion  es_test  anomaly_score  is_anomaly anomaly_label
2026-10-09 19:41:03                 Starting Lynis 3.0.9 with PID 57294, build date 2023-08-03                    0.0         0.0                58           0              0        0       0.106957           1        Normal
2026-10-09 19:41:03                                                                       ====                    0

**Extra:** las 10 líneas más anómalas (puntaje más bajo = más rara).

In [7]:
df.sort_values("anomaly_score").head(10)[["timestamp", "duracion_s", "mensaje", "anomaly_score"]]

,timestamp,duracion_s,mensaje,anomaly_score
7795,2026-10-09 19:45:02,1.0,Suggestion: Update your system with apt-get up...,-0.127907
4014,2026-10-09 19:42:53,9.0,Performing test ID DEB-0280 (Checking if libpa...,-0.118616
10375,2026-10-09 19:46:33,0.0,Suggestion: Harden the system by installing at...,-0.116109
10356,2026-10-09 19:46:32,0.0,Suggestion: One or more sysctl values differ f...,-0.115087
4040,2026-10-09 19:43:03,0.0,"Suggestion: Install needrestart, alternatively...",-0.102822
7830,2026-10-09 19:45:03,7.0,Performing test ID NETW-2600 (Checking IPv6 co...,-0.094955
66,2026-10-09 19:41:12,0.0,Suggestion: This release is more than 4 months...,-0.094150
10240,2026-10-09 19:46:22,2.0,Performing test ID KRNL-6000 (Check sysctl key...,-0.094095
10228,2026-10-09 19:46:21,1.0,Performing test ID HOME-9310 (Checking for sus...,-0.093903
9226,2026-10-09 19:45:46,1.0,Test: check if we can access /usr/share/ca-cer...,-0.091847


**Descargar los CSV** a tu computadora.

In [8]:
from google.colab import files
files.download("lynis_log.csv")
files.download("isolation_forest_resultados.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>